# Piloto: color de los techos de Barranquilla con aprendizaje profundo

Prueba pequeña del módulo opcional del Paso 6 de la propuesta (material y color de los techos con imágenes satelitales y aprendizaje profundo). No busca resultados para la tesis, sino responder tres preguntas:

1. ¿La imagen y los polígonos del catastro quedan bien alineados?
2. ¿Se distinguen a simple vista las clases de color de los techos?
3. ¿Una red convolucional acierta más que el promedio simple de los píxeles de cada techo?

**Versión 2 (20-nov-2026).** Cambios después de la primera corrida:

- Las construcciones del catastro que se tocan y son del mismo predio se unen en un solo techo, y se descartan los techos de menos de 20 m². En los estratos 1 a 3, cerca de un 20 % de las construcciones eran pedazos de una casa.
- Los árboles (con el NDVI) y la sombra se detectan píxel por píxel. El color se mide solo con los píxeles visibles del interior del techo, y un techo con menos de la mitad visible queda como `no_visible` sin etiquetarlo.
- La tesis necesita la absortancia, así que la clase que se predice es el **tono**: claro, medio u oscuro.
- El **color** (blanco, gris, marrón, rojo, verde, azul) va aparte, como demostración de lo que se puede hacer con los datos de Catastro (sección 9).

**Datos.** Seis recortes de 200 × 200 m, uno por estrato (Carrizal, El Milagro, Cevillar, El Recreo, El Prado y El Golf), descargados del servidor de la Gerencia de Gestión Catastral (`miciudad.barranquilla.gov.co`, licencia CC BY-SA 4.0):

- Ortoimagen WorldView Legion del 28 de junio de 2026, 34 cm, 8 bandas de 16 bits.
- Ortofoto aérea de 2016 remuestreada a 25 cm, solo para comparar.
- Polígonos de la capa Construcción y manzanas con su estrato.

**Antes de empezar.**
1. Sube `piloto_techos.tar` a tu Drive, en `MyDrive/piloto_techos/`.
2. En Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4 o L4)*.
3. Corre las celdas en orden. El etiquetado (sección 4) toma entre 30 y 40 minutos y guarda cada etiqueta en Drive, así que puedes pararlo y seguir después.

Semilla 42 en todo.

## 0. Configuración

In [ ]:
import sys, subprocess
# rasterio no viene en Colab; geopandas, scikit-learn y tensorflow sí
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio'], check=True)

from google.colab import drive
drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/piloto_techos'   # donde subiste piloto_techos.tar

In [ ]:
import os, io, json, tarfile, random, pathlib
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio import features
from scipy import ndimage
from shapely.geometry import box
from PIL import Image, ImageDraw
import tensorflow as tf

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); tf.keras.utils.set_random_seed(SEMILLA)

DATOS = pathlib.Path('/content/datos')
RAIZ = DATOS / 'piloto_techos'
if not RAIZ.exists():
    with tarfile.open(f'{CARPETA}/piloto_techos.tar') as t:
        t.extractall(DATOS, filter='data')

ZONAS = json.loads((RAIZ / 'zonas.json').read_text(encoding='utf-8'))['zonas']
NOMBRES = [z[0] for z in ZONAS]
ESTRATO_ZONA = {n: int(n[1]) for n in NOMBRES}
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Zonas:', NOMBRES)

## 1. Carga de los datos

Las 8 bandas de WorldView Legion van en este orden: costera, azul, verde, amarillo, rojo, borde rojo, infrarrojo cercano 1 e infrarrojo cercano 2. El orden se comprobó con la imagen: con las bandas 4, 2 y 1 los árboles se ven verdes, y con la 6 en el canal rojo se ven rojos.

Para pasar a 8 bits se usa **un solo estiramiento para todas las zonas**, entre los percentiles 2 y 98 de cada banda. Así un mismo color se ve igual en todos los recortes.

In [ ]:
BANDAS = ['costera', 'azul', 'verde', 'amarillo', 'rojo', 'borde_rojo', 'nir1', 'nir2']
RGB, ROJO, NIR = (4, 2, 1), 4, 6

img, transf, mas, transf_mas, constr, manz = {}, {}, {}, {}, {}, {}
for n in NOMBRES:
    with rasterio.open(RAIZ / n / 'wv2026_8b.tif') as src:
        img[n] = src.read().astype('float32')          # (8, 588, 588)
        transf[n] = src.transform
        # el GeoTIFF trae el sistema como texto y rasterio no siempre lo traduce a EPSG:9377
        assert src.crs.to_epsg() == 9377 or 'Origen-Nacional' in src.crs.to_wkt(), src.crs
    with rasterio.open(RAIZ / n / 'mas2016_rgb.tif') as src:
        mas[n] = np.moveaxis(src.read()[:3], 0, -1)
        transf_mas[n] = src.transform
    constr[n] = gpd.read_file(RAIZ / n / 'construcciones.geojson').set_crs(9377, allow_override=True)
    manz[n] = gpd.read_file(RAIZ / n / 'manzanas_estrato.geojson').set_crs(9377, allow_override=True)
    print(f'{n}: imagen {img[n].shape}, {len(constr[n])} construcciones')

todo = np.concatenate([img[n].reshape(8, -1) for n in NOMBRES], axis=1)
P2, P98 = np.percentile(todo, 2, axis=1), np.percentile(todo, 98, axis=1)
del todo

def a_uint8(a, idx=RGB):
    return np.dstack([(np.clip((a[i] - P2[i]) / (P98[i] - P2[i]), 0, 1) * 255).astype('uint8') for i in idx])

rgb8 = {n: a_uint8(img[n]) for n in NOMBRES}

def extension(t, H, W):
    return (t.c, t.c + t.a * W, t.f + t.e * H, t.f)   # izquierda, derecha, abajo, arriba

## 2. Revisión visual: bandas y alineación (pregunta 1)

Primero las composiciones de color para la zona de Cevillar. Después cada zona con los polígonos del catastro encima, en la imagen de 2026 y en la de 2016. Revisa sobre todo:

- si los polígonos caen sobre los techos o quedan corridos;
- en El Prado y El Golf, donde hay edificios de 4 pisos o más, si el techo aparece desplazado respecto a la huella (el satélite no mira justo hacia abajo);
- construcciones que existen en el catastro y no en la imagen, o al revés.

In [ ]:
n = 'e3_cevillar'
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a_, (titulo, idx) in zip(ax, [('Color natural (4, 2, 1)', RGB), ('Falso color infrarrojo (6, 4, 2)', (6, 4, 2)), ('Borde rojo (5, 4, 2)', (5, 4, 2))]):
    a_.imshow(a_uint8(img[n], idx)); a_.set_title(titulo); a_.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
for n in NOMBRES:
    H, W = img[n].shape[1:]
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(rgb8[n], extent=extension(transf[n], H, W))
    Hm, Wm = mas[n].shape[:2]
    ax[1].imshow(mas[n], extent=extension(transf_mas[n], Hm, Wm))
    for a_, t in zip(ax, ['WorldView 2026 (34 cm)', 'Ortofoto 2016 (25 cm)']):
        altos = constr[n][constr[n].total_pisos.fillna(0) >= 4]
        constr[n].boundary.plot(ax=a_, color='yellow', linewidth=0.6)
        if len(altos): altos.boundary.plot(ax=a_, color='cyan', linewidth=1.2)
        a_.set_title(f'{n}: {t}'); a_.axis('off')
    plt.tight_layout(); plt.show()
print('Amarillo: construcciones. Cian: 4 pisos o más.')

**Anota aquí lo que ves** (alineación buena o mala, desplazamientos en edificios altos, cambios entre 2016 y 2026):

-

## 3. Techos, máscara de árboles y sombra, y línea base

1. **Unión por predio.** Las construcciones que se tocan y tienen el mismo terreno (los 21 primeros dígitos de `local_id`, el número predial) se unen en un solo techo. Se descartan los techos de menos de 20 m².
2. **Píxeles interiores.** Para medir el color se usan solo los píxeles a más de 0,7 m del borde del polígono. Así los polígonos un poco corridos no toman la calle o el techo vecino.
3. **Árboles y sombra.** Un píxel es árbol si su NDVI pasa de 0,3. Es sombra si es muy oscuro en el visible y también en el infrarrojo, porque un techo oscuro sí refleja algo de infrarrojo. Un techo con menos de la mitad de sus píxeles interiores visibles queda como `no_visible`.
4. **Tono y color automáticos.** Con la mediana de los píxeles visibles, el tono sale del brillo (claro, medio u oscuro) y el color del matiz. Son la línea base más simple, sin aprendizaje.

Además, para cada techo se guardan el promedio de cada banda, el NDVI y las fracciones de árbol, sombra y píxeles visibles, que usa el bosque aleatorio.

In [ ]:
import shapely, colorsys

UMBRAL_NDVI = 0.3     # revisa el histograma de abajo; la imagen no está calibrada en reflectancia
UMBRAL_SOMBRA = 0.07  # brillo visible por debajo del cual un píxel puede ser sombra
UMBRAL_SOMBRA_NIR = 0.12
MIN_M2 = 20           # techos más pequeños se descartan
BORDE_M = 0.7         # los píxeles a menos de esta distancia del borde no se usan para medir
MIN_VISIBLE = 0.5     # fracción mínima de píxeles interiores visibles
CORTES_TONO = (0.30, 0.55)   # brillo mediano: oscuro < 0,30 ≤ medio < 0,55 ≤ claro
TAM = 128             # tamaño de los recortes para la red

def ventana(geom, t, H, W, margen):
    x0, y0, x1, y1 = geom.bounds
    c0 = int(np.floor((x0 - margen - t.c) / t.a)); c1 = int(np.ceil((x1 + margen - t.c) / t.a))
    r0 = int(np.floor((y1 + margen - t.f) / t.e)); r1 = int(np.ceil((y0 - margen - t.f) / t.e))
    return max(r0, 0), min(r1, H), max(c0, 0), min(c1, W)

def unir_por_predio(g):
    """Une las construcciones que se tocan y son del mismo terreno; guarda qué construcciones forman cada techo."""
    g = g.copy()
    g['predio'] = g.local_id.astype(str).str[:21]
    filas = []
    for predio, sub in g.groupby('predio'):
        u = shapely.union_all(list(sub.geometry.buffer(0.3))).buffer(-0.3)
        partes = list(u.geoms) if u.geom_type == 'MultiPolygon' else [u]
        for k, p in enumerate(partes):
            if p.is_empty: continue
            dentro = sub[sub.geometry.representative_point().within(p.buffer(0.5))]
            filas.append({'predio': predio, 'parte': k, 'geometry': p,
                          'objectids': [int(o) for o in dentro.objectid],
                          'n_construcciones': len(dentro),
                          'total_pisos': dentro.total_pisos.max() if len(dentro) else np.nan})
    return gpd.GeoDataFrame(filas, geometry='geometry', crs=g.crs)

def nombre_color(r, g_, b):
    """Nombre del color a partir de la mediana RGB (0 a 1) de los píxeles visibles."""
    h, s, v = colorsys.rgb_to_hsv(r, g_, b); h *= 360
    if v > 0.72 and s < 0.18: return 'blanco'
    if (h < 20 or h > 345) and s > 0.35 and v > 0.3: return 'rojo_terracota'
    if 75 <= h < 165 and s > 0.15 and v > 0.3: return 'verde'
    if 165 <= h < 255 and s > 0.15 and v > 0.3: return 'azul'
    if 15 <= h < 60 and (s > 0.2 or (v < 0.4 and s > 0.1)): return 'marron'
    return 'gris_claro' if v > 0.45 else 'gris_oscuro'

def tono_de(brillo):
    return 'oscuro' if brillo < CORTES_TONO[0] else ('medio' if brillo < CORTES_TONO[1] else 'claro')

tablas, recortes, etiquetas_px, mascaras = [], [], {}, {}
for n in NOMBRES:
    a, t = img[n], transf[n]
    H, W = a.shape[1:]
    left, right, bottom, top = extension(t, H, W)
    dentro_caja = constr[n][constr[n].within(box(left + 1, bottom + 1, right - 1, top - 1))]
    g = unir_por_predio(dentro_caja)
    g = g[g.area >= MIN_M2].reset_index(drop=True)
    idx = np.arange(1, len(g) + 1)
    ids = features.rasterize(((geom, i + 1) for i, geom in enumerate(g.geometry)),
                             out_shape=(H, W), transform=t, fill=0, dtype='int32')
    interior = [geom.buffer(-BORDE_M) if not geom.buffer(-BORDE_M).is_empty else geom for geom in g.geometry]
    ids_int = features.rasterize(((geom, i + 1) for i, geom in enumerate(interior)),
                                 out_shape=(H, W), transform=t, fill=0, dtype='int32')
    etiquetas_px[n] = ids

    esc = np.stack([(a[b] - P2[b]) / (P98[b] - P2[b]) for b in range(8)])
    ndvi_px = (a[NIR] - a[ROJO]) / (a[NIR] + a[ROJO] + 1e-6)
    brillo_px = (esc[RGB[0]] + esc[RGB[1]] + esc[RGB[2]]) / 3
    veg = ndvi_px > UMBRAL_NDVI
    sombra = (brillo_px < UMBRAL_SOMBRA) & (esc[NIR] < UMBRAL_SOMBRA_NIR) & ~veg
    mascaras[n] = (veg, sombra)
    ids_vis = np.where(veg | sombra, 0, ids_int)

    uno = np.ones_like(ids)
    g['n_px'] = ndimage.sum(uno, ids, idx)
    n_int = ndimage.sum(uno, ids_int, idx)
    g['frac_veg'] = ndimage.sum(veg.astype('float32'), ids_int, idx) / np.maximum(n_int, 1)
    g['frac_sombra'] = ndimage.sum(sombra.astype('float32'), ids_int, idx) / np.maximum(n_int, 1)
    g['frac_visible'] = ndimage.sum(uno, ids_vis, idx) / np.maximum(n_int, 1)
    hay = g.frac_visible.values > 0
    for b in range(8):
        m_vis = np.asarray(ndimage.mean(esc[b], ids_vis, idx), dtype='float32')
        m_int = np.asarray(ndimage.mean(esc[b], ids_int, idx), dtype='float32')
        g[f'm_{BANDAS[b]}'] = np.where(hay, m_vis, m_int)
    g['ndvi'] = ndimage.mean(ndvi_px, ids_int, idx)
    g['brillo'] = np.where(hay, ndimage.median(brillo_px, ids_vis, idx), ndimage.median(brillo_px, ids_int, idx))
    g['brillo_sd'] = ndimage.standard_deviation(brillo_px, ids_vis, idx)
    med_rgb = np.stack([np.where(hay, ndimage.median(esc[i], ids_vis, idx), np.nan) for i in RGB], axis=1)
    visible = g.frac_visible.values >= MIN_VISIBLE
    g['tono_auto'] = [tono_de(b) if v else 'no_visible' for b, v in zip(g.brillo.values, visible)]
    g['color_auto'] = [nombre_color(*np.clip(c, 0, 1)) if v else 'no_visible' for c, v in zip(med_rgb, visible)]
    g['id_px'] = idx
    g['zona'] = n
    # estrato de la manzana que contiene el punto interior del techo
    pts = g[['geometry']].copy(); pts['geometry'] = g.representative_point()
    union = gpd.sjoin(pts, manz[n][['estratificacion', 'geometry']], how='left', predicate='within')
    union = union[~union.index.duplicated()]
    g['estrato_manzana'] = union['estratificacion'].reindex(g.index).values
    g = g[g.n_px >= 30]
    # recortes RGB: el techo a color y el entorno atenuado, para que la red sepa cuál es
    for _, f in g.iterrows():
        r0, r1, c0, c1 = ventana(f.geometry, t, H, W, margen=2)
        sub = rgb8[n][r0:r1, c0:c1].astype('float32')
        sub[ids[r0:r1, c0:c1] != f.id_px] *= 0.35
        recortes.append(np.asarray(Image.fromarray(sub.astype('uint8')).resize((TAM, TAM), Image.BILINEAR)))
    tablas.append(g)

techos = gpd.GeoDataFrame(pd.concat(tablas, ignore_index=True), geometry='geometry', crs=9377)
techos['clave'] = techos.zona + '_' + techos.predio + '_' + techos.parte.astype(str)
techos['estrato_zona'] = techos.zona.map(ESTRATO_ZONA)
X = np.stack(recortes)
assert len(X) == len(techos) and techos.clave.is_unique
print(f'{len(techos)} techos (construcciones unidas por predio); recortes {X.shape}')
display(pd.crosstab(techos.zona, techos.tono_auto, margins=True, margins_name='total'))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(np.concatenate([((img[n][NIR] - img[n][ROJO]) / (img[n][NIR] + img[n][ROJO] + 1e-6)).ravel() for n in NOMBRES]), bins=100)
ax[0].axvline(UMBRAL_NDVI, color='red'); ax[0].set_title('NDVI de todos los píxeles (línea: umbral de árboles)')
ax[1].hist(techos.brillo, bins=60)
for c in CORTES_TONO: ax[1].axvline(c, color='red')
ax[1].set_title('Brillo mediano de cada techo (líneas: cortes del tono)')
plt.tight_layout(); plt.show()

# máscara de árboles (verde) y sombra (morado) en dos zonas, para revisar los umbrales
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for j, n in enumerate(['e3_cevillar', 'e6_el_golf']):
    veg, sombra = mascaras[n]
    over = rgb8[n].copy(); over[veg] = (50, 215, 50); over[sombra] = (190, 50, 215)
    ax[2 * j].imshow(rgb8[n]); ax[2 * j].set_title(f'{n}: color natural')
    ax[2 * j + 1].imshow(over); ax[2 * j + 1].set_title('verde: árboles · morado: sombra')
for a_ in ax: a_.axis('off')
plt.tight_layout(); plt.show()

## 4. Etiquetado (pregunta 2)

Para cada techo hay **dos filas de botones**:

1. **Color** (opcional, para la demostración de la sección 9): haz clic en el color y *no* avanza.
2. **Tono** (obligatorio, es lo que necesita la tesis): haz clic en el tono y pasa al siguiente techo.

| Tono | Qué incluye |
|---|---|
| `claro` | Blanco, pintado de claro, aluminio brillante |
| `medio` | Teja roja o terracota, azul claro, verde claro, zinc en buen estado, gris medio |
| `oscuro` | Gris oscuro, marrón oscuro, verde o azul oscuro, zinc oxidado, asbesto o losa envejecida |
| `no_visible` | Úsalo solo si la máscara no lo detectó: más de la mitad tapada, o el polígono cae fuera del techo |

Se ven tres imágenes: el techo a color con el entorno atenuado, la máscara (verde: árboles, morado: sombra) y el entorno con el polígono en amarillo. Reglas: etiqueta lo que ves en **2026**. Si el techo mezcla tonos, elige el de la mayor parte. Si dudas, usa **saltar**.

Las etiquetas de la primera versión se reaprovechan: cuando todas las construcciones que forman un techo tenían la misma clase, el tono pasa solo (`claro_blanco` → claro, `gris_metalico` y `teja_roja` → medio, `gris_oscuro` → oscuro). Las de `pintado_verde_azul` y `no_visible` se vuelven a etiquetar. Las etiquetas nuevas se guardan en `etiquetas_v2.csv`, sin tocar el archivo anterior.

Los techos salen alternando zonas, y dentro de cada zona alternan los que el tono automático ve como oscuros y los que no, para que las clases pequeñas tengan más ejemplos.

In [ ]:
import ipywidgets as w
from IPython.display import display

TONOS = ['claro', 'medio', 'oscuro', 'no_visible']
COLORES_ETQ = ['blanco', 'gris_claro', 'gris_oscuro', 'marron', 'rojo_terracota', 'verde', 'azul', 'otro']
META = 250
ETQ = f'{CARPETA}/etiquetas_v2.csv'
ETQ_V1 = f'{CARPETA}/etiquetas_piloto.csv'

# etiquetas nuevas: clave -> {'tono', 'color', 'origen'}
etq = {}
if os.path.exists(ETQ):
    etq = {r.clave: {'tono': r.tono, 'color': (r.color if isinstance(r.color, str) else None), 'origen': r.origen}
           for r in pd.read_csv(ETQ).itertuples()}

# herencia de la primera versión: solo si todas las construcciones del techo tenían la misma clase
TONO_V1 = {'claro_blanco': 'claro', 'gris_metalico': 'medio', 'teja_roja': 'medio', 'gris_oscuro': 'oscuro'}
COLOR_V1 = {'claro_blanco': 'blanco', 'gris_metalico': 'gris_claro', 'teja_roja': 'rojo_terracota', 'gris_oscuro': 'gris_oscuro'}
heredadas = 0
if os.path.exists(ETQ_V1):
    v1 = pd.read_csv(ETQ_V1).set_index('clave')['clase'].to_dict()
    for f in techos.itertuples():
        if f.clave in etq: continue
        clases = [v1.get(f'{f.zona}_{o}') for o in f.objectids]
        clases = [c for c in clases if c is not None]
        if clases and len(set(clases)) == 1 and clases[0] in TONO_V1:
            etq[f.clave] = {'tono': TONO_V1[clases[0]], 'color': COLOR_V1[clases[0]], 'origen': 'v1'}
            heredadas += 1
print(f'Heredadas de la primera versión: {heredadas}. Etiquetas en total: {len(etq)}')

rng = np.random.default_rng(SEMILLA)
por_zona = {}
for n in NOMBRES:
    ks = rng.permutation(np.flatnonzero(techos.zona.values == n))
    osc = [int(k) for k in ks if techos.tono_auto.iloc[k] == 'oscuro']
    otros = [int(k) for k in ks if techos.tono_auto.iloc[k] != 'oscuro']
    mezcla = []
    while osc or otros:
        if otros: mezcla.append(otros.pop(0))
        if osc: mezcla.append(osc.pop(0))
    por_zona[n] = mezcla
orden = []
while any(por_zona.values()):
    for n in NOMBRES:
        if por_zona[n]: orden.append(por_zona[n].pop(0))

def png_bytes(arr, ancho=256):
    im = Image.fromarray(arr)
    im = im.resize((ancho, int(ancho * im.height / im.width)), Image.NEAREST)
    buf = io.BytesIO(); im.save(buf, format='PNG'); return buf.getvalue()

def contexto(k, margen=12, con_mascara=False):
    f = techos.iloc[k]; n = f.zona; t = transf[n]; H, W = rgb8[n].shape[:2]
    r0, r1, c0, c1 = ventana(f.geometry, t, H, W, margen)
    base = rgb8[n][r0:r1, c0:c1].copy()
    if con_mascara:
        veg, sombra = mascaras[n]
        base[veg[r0:r1, c0:c1]] = (50, 215, 50); base[sombra[r0:r1, c0:c1]] = (190, 50, 215)
    im = Image.fromarray(base).resize(((c1 - c0) * 3, (r1 - r0) * 3), Image.NEAREST)
    d = ImageDraw.Draw(im)
    polis = f.geometry.geoms if f.geometry.geom_type == 'MultiPolygon' else [f.geometry]
    for p in polis:
        xy = [(((x - t.c) / t.a - c0) * 3, ((y - t.f) / t.e - r0) * 3) for x, y in p.exterior.coords]
        d.line(xy + [xy[0]], fill=(255, 255, 0), width=2)
    return np.asarray(im)

def guardar():
    os.makedirs(CARPETA, exist_ok=True)
    tabla_etq = pd.DataFrame([{'clave': k, **v} for k, v in etq.items()], columns=['clave', 'tono', 'color', 'origen'])
    tabla_etq.to_csv(ETQ, index=False)
    tabla_etq.to_csv('/content/etiquetas_v2_respaldo.csv', index=False)

def siguiente(desde):
    for j in range(desde, len(orden)):
        if techos.clave.iloc[orden[j]] not in etq:
            return j
    return None

guardar()
estado = {'i': siguiente(0), 'hist': [], 'color': None}
im_techo, im_masc, im_ctx = (w.Image(format='png', width=230) for _ in range(3))
info, prog, sel = w.HTML(), w.HTML(), w.HTML()

def mostrar():
    j = estado['i']
    if j is None:
        info.value = '<b>No quedan techos sin etiquetar.</b>'; return
    k = orden[j]; f = techos.iloc[k]
    im_techo.value = png_bytes(X[k]); im_masc.value = png_bytes(contexto(k, con_mascara=True))
    im_ctx.value = png_bytes(contexto(k))
    info.value = (f'<b>{f.zona}</b> · predio {f.predio} · {f.n_construcciones} construcción(es) · '
                  f'pisos: {f.total_pisos} · área: {f.geometry.area:.0f} m² · visible: {f.frac_visible:.0%} · '
                  f'automático: {f.tono_auto} / {f.color_auto}')
    estado['color'] = None; sel.value = 'Color: —'
    nuevas = sum(v['origen'] == 'manual' for v in etq.values())
    prog.value = (f'Etiquetas: <b>{len(etq)}</b> ({nuevas} nuevas) / meta {META}'
                  + (' · <b style="color:green">meta alcanzada</b>' if len(etq) >= META else ''))

errores = w.Output()   # los errores dentro de los botones no se ven si no se capturan aquí

def clic_color(b):
    with errores:
        estado['color'] = b.description; sel.value = f'Color: <b>{b.description}</b>'

def clic_tono(b):
    with errores:
        j = estado['i']
        if j is None: return
        clave = techos.clave.iloc[orden[j]]
        if b.description != 'saltar':
            etq[clave] = {'tono': b.description, 'color': estado['color'], 'origen': 'manual'}; guardar()
        estado['hist'].append(j); estado['i'] = siguiente(j + 1); mostrar()

def atras(_):
    with errores:
        if estado['hist']:
            j = estado['hist'].pop(); etq.pop(techos.clave.iloc[orden[j]], None); guardar()
            estado['i'] = j; mostrar()

b_col = [w.Button(description=c, layout=w.Layout(width='120px')) for c in COLORES_ETQ]
b_ton = [w.Button(description=c, button_style='success', layout=w.Layout(width='140px')) for c in TONOS]
b_saltar = w.Button(description='saltar', button_style='warning'); b_atras = w.Button(description='atrás', button_style='info')
for b in b_col: b.on_click(clic_color)
for b in b_ton + [b_saltar]: b.on_click(clic_tono)
b_atras.on_click(atras)

display(w.VBox([prog, info, w.HBox([im_techo, im_masc, im_ctx]),
                w.HTML('<b>1. Color</b> (opcional)'), w.HBox(b_col[:4]), w.HBox(b_col[4:]), sel,
                w.HTML('<b>2. Tono</b> (pasa al siguiente)'), w.HBox(b_ton), w.HBox([b_saltar, b_atras]), errores]))
mostrar()
# No uses «Ejecutar todo»: etiqueta aquí y corre la celda siguiente solo cuando termines.

In [ ]:
if os.path.exists(ETQ):
    etq = {r.clave: {'tono': r.tono, 'color': (r.color if isinstance(r.color, str) else None), 'origen': r.origen}
           for r in pd.read_csv(ETQ).itertuples()}
assert len(etq) > 0, 'Todavía no hay etiquetas: etiqueta techos en la celda anterior antes de correr esta.'
techos['tono'] = techos.clave.map({k: v['tono'] for k, v in etq.items()})
techos['color'] = techos.clave.map({k: v['color'] for k, v in etq.items()})
lab = techos.dropna(subset=['tono'])
print(f'{len(lab)} techos con tono ({(lab.clave.map(lambda k: etq[k]["origen"]) == "v1").sum()} heredados)')
display(pd.crosstab(lab.tono, lab.zona, margins=True, margins_name='total'))
print('Tono a mano frente al automático (solo brillo, sin aprendizaje):')
display(pd.crosstab(lab.tono, lab.tono_auto, margins=True, margins_name='total'))

fig, ax = plt.subplots(len(TONOS), 8, figsize=(16, 2.1 * len(TONOS)))
for r, c in enumerate(TONOS):
    ks = np.flatnonzero(techos.tono.values == c)[:8]
    for j in range(8):
        ax[r, j].axis('off')
        if j < len(ks): ax[r, j].imshow(X[ks[j]])
    ax[r, 0].set_title(c, loc='left', fontsize=10)
plt.tight_layout(); plt.show()

**Anota aquí** qué tonos se distinguen bien, cuáles se confunden, si la máscara de sombra marca techos oscuros por error y si el tono automático ya acierta:

-

## 5. Líneas base: tono automático y bosque aleatorio

Hay dos líneas base. La más simple es el **tono automático**, que solo corta el brillo mediano de los píxeles visibles y no aprende nada. La otra es un **bosque aleatorio** con los promedios de las bandas, el NDVI y las fracciones de árbol, sombra y píxeles visibles.

Para medir sin hacer trampa se entrena con cinco zonas y se prueba en la sexta, y se repite para cada zona. Así el modelo nunca ve techos vecinos de los que evalúa. Los techos `no_visible` no entran, porque esa clase ya la decide la máscara. Las clases con menos de 5 etiquetas quedan fuera.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

cuenta = lab.tono.value_counts()
VALIDAS = [c for c in ['claro', 'medio', 'oscuro'] if cuenta.get(c, 0) >= 5]
print('Tonos en la comparación:', VALIDAS)
m = techos.tono.isin(VALIDAS).values
y = techos.loc[m, 'tono'].map({c: i for i, c in enumerate(VALIDAS)}).values
grupos = techos.loc[m, 'zona'].values
ETQS = list(range(len(VALIDAS)))
VARS = [f'm_{b}' for b in BANDAS] + ['ndvi', 'frac_veg', 'frac_sombra', 'frac_visible', 'brillo', 'brillo_sd']
Xf = techos.loc[m, VARS].fillna(0).values
logo = LeaveOneGroupOut()

# tono automático: si la máscara lo vio no visible, cuenta como error (-1)
pred_auto = techos.loc[m, 'tono_auto'].map({c: i for i, c in enumerate(VALIDAS)}).fillna(-1).astype(int).values

pred_rf = np.full(len(y), -1)
for tr, te in logo.split(Xf, y, grupos):
    rf = RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=SEMILLA, n_jobs=-1)
    rf.fit(Xf[tr], y[tr]); pred_rf[te] = rf.predict(Xf[te])

def resumen(pred, nombre):
    return {'modelo': nombre, 'exactitud': accuracy_score(y, pred),
            'exactitud_balanceada': balanced_accuracy_score(y, pred),
            'F1_macro': f1_score(y, pred, average='macro', labels=ETQS, zero_division=0)}
print(resumen(pred_auto, 'tono automático'))
print(resumen(pred_rf, 'bosque aleatorio'))

## 6. Red convolucional con transfer learning (ResNet50)

ResNet50 preentrenada con ImageNet y congelada. Encima, una capa nueva con las clases. Recibe los recortes RGB de 128 × 128 con el entorno atenuado. Para aumentar los datos se voltean y rotan los recortes, porque un techo no tiene una orientación fija.

Con tan pocos techos no hay conjunto de validación dentro de cada pliegue, así que se entrena un número fijo de épocas (30). Las mismas cinco zonas de entrenamiento y la misma zona de prueba que en la línea base.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
pre = tf.keras.applications.resnet50.preprocess_input
Xc = pre(X[m].astype('float32'))
EPOCAS = 30

def crear_modelo(n_clases):
    base = tf.keras.applications.ResNet50(include_top=False, weights='imagenet', input_shape=(TAM, TAM, 3), pooling='avg')
    base.trainable = False
    ent = tf.keras.Input((TAM, TAM, 3))
    x = tf.keras.layers.RandomFlip('horizontal_and_vertical')(ent)
    x = tf.keras.layers.RandomRotation(0.25, fill_mode='reflect')(x)
    x = base(x, training=False)
    x = tf.keras.layers.Dropout(0.3)(x)
    sal = tf.keras.layers.Dense(n_clases, activation='softmax')(x)
    mod = tf.keras.Model(ent, sal)
    mod.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return mod

def pesos(yt):
    # una clase puede faltar en el entrenamiento si solo aparece en la zona de prueba
    cl = np.unique(yt)
    w_ = dict(zip(cl.tolist(), compute_class_weight('balanced', classes=cl, y=yt)))
    return {i: float(w_.get(i, 1.0)) for i in range(len(VALIDAS))}

pred_cnn = np.full(len(y), -1)
for tr, te in logo.split(Xc, y, grupos):
    tf.keras.utils.set_random_seed(SEMILLA)
    mod = crear_modelo(len(VALIDAS))
    mod.fit(Xc[tr], y[tr], epochs=EPOCAS, batch_size=16, class_weight=pesos(y[tr]), verbose=0)
    pred_cnn[te] = mod.predict(Xc[te], verbose=0).argmax(1)
    print(f'zona de prueba {grupos[te][0]}: exactitud {accuracy_score(y[te], pred_cnn[te]):.2f}')
print(resumen(pred_cnn, 'ResNet50'))

## 7. Comparación (pregunta 3)

Con unos 250 techos la diferencia es solo orientativa. El intervalo de la diferencia de F1 macro entre la red y el bosque se calcula con remuestreo (bootstrap) de los techos.

In [ ]:
PRED = {'tono automático': pred_auto, 'bosque aleatorio': pred_rf, 'ResNet50': pred_cnn}
tabla = pd.DataFrame([resumen(p, k) for k, p in PRED.items()]).set_index('modelo').round(3)
display(tabla)

por_zona_tab = pd.DataFrame({'zona': grupos, **{k: p == y for k, p in PRED.items()}}).groupby('zona').mean().round(2)
por_zona_tab['n'] = pd.Series(grupos).value_counts()
display(por_zona_tab)

rng = np.random.default_rng(SEMILLA)
dif = []
for _ in range(2000):
    s = rng.integers(0, len(y), len(y))
    dif.append(f1_score(y[s], pred_cnn[s], average='macro', labels=ETQS, zero_division=0)
               - f1_score(y[s], pred_rf[s], average='macro', labels=ETQS, zero_division=0))
lo, hi = np.percentile(dif, [2.5, 97.5])
print(f'F1 macro (ResNet50 − bosque): {tabla.F1_macro["ResNet50"] - tabla.F1_macro["bosque aleatorio"]:+.3f}, '
      f'IC 95 % [{lo:+.3f}, {hi:+.3f}]')

fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))
for a_, (t_, p) in zip(ax, PRED.items()):
    ConfusionMatrixDisplay(confusion_matrix(y, p, labels=ETQS), display_labels=VALIDAS).plot(ax=a_, colorbar=False)
    a_.set_title(t_)
plt.tight_layout(); plt.show()

## 8. Proporción de tonos por estrato

Se entrena el modelo con mejor F1 macro usando todas las etiquetas y se aplica a todos los techos de las seis zonas; los `no_visible` los decide la máscara. Como todo modelo se equivoca, las proporciones se corrigen con su matriz de confusión de la validación por zona: si el modelo confunde, por ejemplo, el 20 % de los techos medios con oscuros, la corrección devuelve esa parte. Solo se guardan proporciones por estrato, no resultados por vivienda.

In [ ]:
from scipy.optimize import nnls

mejor = tabla.F1_macro.idxmax()
if mejor == 'ResNet50':
    tf.keras.utils.set_random_seed(SEMILLA)
    mod = crear_modelo(len(VALIDAS)); mod.fit(Xc, y, epochs=EPOCAS, batch_size=16, class_weight=pesos(y), verbose=0)
    pred_todos = np.array(VALIDAS)[mod.predict(pre(X.astype('float32')), verbose=0).argmax(1)]
elif mejor == 'bosque aleatorio':
    rf = RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=SEMILLA, n_jobs=-1).fit(Xf, y)
    pred_todos = np.array(VALIDAS)[rf.predict(techos[VARS].fillna(0).values)]
else:
    pred_todos = techos.tono_auto.values
techos['pred'] = np.where(techos.tono_auto.values == 'no_visible', 'no_visible', pred_todos)
print('Modelo aplicado:', mejor)

# matriz P(predicho | real) de la validación por zona
Cm = confusion_matrix(y, PRED[mejor], labels=ETQS).astype(float)
Cm = Cm / np.maximum(Cm.sum(1, keepdims=True), 1)

def corregir(obs):
    p, _ = nnls(Cm.T, obs)
    return p / p.sum() if p.sum() > 0 else obs

filas = []
for e, sub in techos.groupby('estrato_zona'):
    vis = sub[sub.pred != 'no_visible']
    obs = np.array([(vis.pred == c).mean() for c in VALIDAS])
    cor = corregir(obs)
    fila = {'estrato': e, 'techos': len(sub), 'no_visible': round((sub.pred == 'no_visible').mean(), 3)}
    fila.update({f'{c}_observado': round(o, 3) for c, o in zip(VALIDAS, obs)})
    fila.update({f'{c}_corregido': round(c_, 3) for c, c_ in zip(VALIDAS, cor)})
    filas.append(fila)
prop = pd.DataFrame(filas).set_index('estrato')
display(prop)
prop.to_csv(f'{CARPETA}/proporciones_tono_por_estrato_piloto.csv')

COL_TONO = {'claro': '#f2f2f2', 'medio': '#c08a5a', 'oscuro': '#3d3d3d', 'no_visible': '#ff00ff'}
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
for a_, n in zip(ax.ravel(), NOMBRES):
    H, W = img[n].shape[1:]
    a_.imshow(rgb8[n], extent=extension(transf[n], H, W))
    sub = techos[techos.zona == n]
    sub.plot(ax=a_, color=sub.pred.map(COL_TONO).tolist(), edgecolor='yellow', linewidth=0.3, alpha=0.75)
    a_.set_title(n); a_.axis('off')
plt.tight_layout(); plt.show()

json.dump({'version': 2, 'n_techos': int(len(techos)), 'n_etiquetas': int(len(lab)), 'tonos': VALIDAS,
           'metricas': tabla.reset_index().to_dict('records'),
           'dif_F1_cnn_menos_bosque_IC95': [float(lo), float(hi)], 'modelo_aplicado': mejor},
          open(f'{CARPETA}/resultados_piloto_v2.json', 'w'), indent=1, ensure_ascii=False)
print('Guardado en Drive: proporciones_tono_por_estrato_piloto.csv y resultados_piloto_v2.json')

## 9. Demostración para Catastro: color de los techos

Esta sección no entra en la tesis. Muestra lo que se puede obtener combinando la ortoimagen WorldView 2026 con las construcciones del catastro: el color de cada techo, sin entrenar ningún modelo, a partir del color mediano de sus píxeles visibles. Si etiquetaste colores, se compara el color automático con el tuyo.

La imagen tiene balance radiométrico pero no está calibrada en reflectancia, así que los nombres de color son relativos a esta imagen. Las figuras muestran construcciones, no viviendas, y no deben publicarse con la dirección o el número predial de cada una.

In [ ]:
COL_COLOR = {'blanco': '#f4f4f4', 'gris_claro': '#a8adb3', 'gris_oscuro': '#3b3b3b', 'marron': '#8b5a2b',
             'rojo_terracota': '#c0502b', 'verde': '#3e8e41', 'azul': '#3c78d8', 'no_visible': '#ff00ff'}
from matplotlib.patches import Patch

fig, ax = plt.subplots(2, 3, figsize=(16, 11))
for a_, n in zip(ax.ravel(), NOMBRES):
    H, W = img[n].shape[1:]
    a_.imshow((rgb8[n] * 0.55).astype('uint8'), extent=extension(transf[n], H, W))
    sub = techos[techos.zona == n]
    sub.plot(ax=a_, color=sub.color_auto.map(COL_COLOR).tolist(), edgecolor='yellow', linewidth=0.3, alpha=0.9)
    a_.set_title(n); a_.axis('off')
fig.legend([Patch(fc=v) for v in COL_COLOR.values()], list(COL_COLOR), loc='lower center', ncol=8)
fig.suptitle('Color de los techos estimado con la ortoimagen WorldView 2026 y las construcciones del catastro')
plt.tight_layout(rect=(0, 0.04, 1, 0.97), h_pad=2.5)
plt.savefig(f'{CARPETA}/demo_color_techos.png', dpi=150); plt.show()

tab_color = pd.crosstab(techos.estrato_zona, techos.color_auto, normalize='index').round(3)
display(tab_color)
tab_color.to_csv(f'{CARPETA}/demo_color_por_estrato.csv')

con_color = techos.dropna(subset=['color'])
con_color = con_color[con_color.color_auto != 'no_visible']
if len(con_color):
    print(f'Color automático frente al etiquetado a mano ({len(con_color)} techos): '
          f'coincide en {(con_color.color == con_color.color_auto).mean():.0%}')
    display(pd.crosstab(con_color.color, con_color.color_auto, margins=True, margins_name='total'))

## 10. Conclusiones del piloto

1. **Alineación y unión por predio:**
2. **Máscara de árboles y sombra:**
3. **Tonos:**
4. **Tono automático, bosque y red:**
5. **Demostración de color para Catastro:**
6. **Qué cambiar antes del estudio completo:**